# 17. Development ranges

Forecast, Phase 1, step 2b. Notebook 15 gives every player one number: his projected WAR per 60 for next season. Real seasons land above or below it. A season simulator needs to know **how far, how often, and for whom**: a 20-year-old top pick can break out, a 34-year-old can fall off a cliff, and anyone can have a lucky or unlucky year.

The ranges come from our own history. For every player-season in notebook 15's backtest (14 target seasons), the **miss** (or residual) is

> miss = actual WAR per 60 − projected WAR per 60

Positive = he beat the projection (a breakout or a lucky year), negative = he fell short. This notebook:

1. Builds the table of misses, with age, NHL experience and draft position.
2. Splits a miss into **luck** (shrinks with more ice time) and **real change** (doesn't).
3. Checks how much of a surprise **sticks** into the next season.
4. Measures misses by age, experience and draft position, and checks which differences are bigger than noise.
5. Asks whether pre-NHL scoring (NHLe, notebook 16) tells us anything extra about young players.
6. Counts the players who **drop out** (and so never show up as a miss).
7. **Tests the ranges on later seasons** (the real check).
8. Sketches how the simulator will draw a season, with a small demo.

The code is in `src/development.py`. Results stay private (`data/dev_residuals.csv`, `data/dev_ranges.csv`, gitignored); nothing on the cards, the app or the README changes.

## 1. Setup and the table of misses

In [1]:
import sys
sys.path.append("../src")

import numpy as np
import pandas as pd
from pathlib import Path

from forecast import history_table
from edge_data import build_bio_table
from nhle import load_profiles
from development import (residual_table, luck_fit, luck_share, expected_sd, add_z, group_means, persistence,
                         bucket_stats, merge_map, best_nhle, wls, dropouts, calibrate, coverage, calibration_score,
                         draw_pools, draw, age_bucket, draft_bucket, wmean, wsd, wcorr, wquantile, cutoff,
                         TARGETS, AGE_ORDER, EXP_ORDER, DRAFT_ORDER, SEASON_GAMES, SK_REG, G_REG, AGING)

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 220)

DATA = Path("../data")
OUT = DATA                      # where the two CSV files are saved

def label(s):
    return f"{s}-{str(s + 1)[-2:]}"

NAMES = {"F": "Forwards", "D": "Defensemen", "G": "Goalies"}
ONE = {"F": "A forward", "D": "A defenseman", "G": "A goalie"}

history = history_table(pd.read_csv(DATA / "war_by_season_2008_2025.csv"), pd.read_csv(DATA / "goalie_war_2008_2025.csv"))
bios = build_bio_table(DATA / "bio_raw", 2026)          # birth dates and draft picks
profiles = load_profiles(DATA / "bio_raw")              # every season in every league (pre-2008 NHL games)
careers = pd.read_csv(DATA / "career_totals.csv")       # notebook 16
factors = pd.read_csv(DATA / "nhle_factors.csv")
multipliers = pd.read_csv(DATA / "nhle_age_multipliers.csv")

print("Projection settings (notebook 15): skaters (1, 1, 1) + {:,} regression minutes, goalies (1, 1, 1) + {:,}; aging {}"
      .format(SK_REG, G_REG, AGING))
print("Target seasons:", ", ".join(label(t) for t in TARGETS))

Projection settings (notebook 15): skaters (1, 1, 1) + 1,500 regression minutes, goalies (1, 1, 1) + 12,000; aging {'F': 'adjust_then_blend', 'D': 'adjust_then_blend', 'G': 'one_year'}
Target seasons: 2009-10, 2010-11, 2011-12, 2013-14, 2014-15, 2015-16, 2016-17, 2017-18, 2018-19, 2021-22, 2022-23, 2023-24, 2024-25, 2025-26


**The table.** One row per player-season the backtest tested: 300+ minutes (skaters) or 1,000+ (goalies) in the target season, and some NHL time the season before. For each:

- **Age** in the target season (February 1).
- **NHL experience:** NHL seasons before the target season with 200+ minutes (skaters) or 500+ (goalies). Before 2008-09 there are no minutes, so the profiles' games stand in: 15+ games (skaters) or 10+ (goalies), which matches 200+ minutes 97% of the time on later seasons. Short seasons (1994-95, 2012-13, 2019-20, 2020-21) get prorated cutoffs. The first bucket is "0-1": a player can be tested after a season under 200 minutes.
- **Draft pick** from the profile; missing = undrafted.
- **The miss** in WAR per 60, and the same miss in WAR at his actual minutes (for scale).

In [2]:
res = residual_table(history, bios, profiles)
print(f"{len(res):,} tested player-seasons: " + ", ".join(f"{NAMES[g]} {n:,}" for g, n in res["group"].value_counts().reindex(["F", "D", "G"]).items()))

# Checks: experience can't be negative or more than his years since age 17; everyone has an age
assert (res["experience"] >= 0).all() and (res["experience"] <= res["age"] - 17).all()
assert res["age"].notna().all()
print("Checks passed: experience between 0 and (age - 17) for everyone, no missing ages")

# The projection's overall bias: the average miss per group (minutes-weighted)
bias = res.groupby("group").apply(lambda d: pd.Series({
    "player-seasons": len(d), "average miss (WAR/60)": wmean(d["residual"], d["toi"]),
    "per 1,000 minutes (WAR)": wmean(d["residual"], d["toi"]) * 1000 / 60}), include_groups=False).reindex(["F", "D", "G"])
bias.index = bias.index.map(NAMES)
print("\nAverage miss by group (negative = the projection is a little too high on average):")
bias.round(4)

9,343 tested player-seasons: Forwards 5,523, Defensemen 3,039, Goalies 781
Checks passed: experience between 0 and (age - 17) for everyone, no missing ages

Average miss by group (negative = the projection is a little too high on average):


,player-seasons,average miss (WAR/60),"per 1,000 minutes (WAR)"
group,,,
Forwards,5523.0,-0.0050,-0.0829
Defensemen,3039.0,-0.0037,-0.0612
Goalies,781.0,0.0001,0.0010


**Read the bias:** skaters come in slightly below their projection on average (notebook 15 found the same: the league's skater WAR level swings from year to year, and minutes go to players coming off good seasons). For the simulator, each group's misses are **centered** (this average subtracted) before drawing, so the league as a whole doesn't drift down every simulated season. Buckets keep their difference from their group's average, so a young player still leans up and an old one down.

**Hockey check: the biggest surprises in history**, in WAR at the player's actual minutes. Skaters and goalies separately, because goalie seasons swing much more in WAR.

In [3]:
show = ["name", "season", "age", "group", "toi", "proj_war_60", "actual_war_60", "residual_war"]
def top(d, n, largest):
    t = (d.nlargest(n, "residual_war") if largest else d.nsmallest(n, "residual_war"))[show].copy()
    t["season"] = t["season"].map(label)
    return t.round(3).reset_index(drop=True)

sk, gk = res[res["group"] != "G"], res[res["group"] == "G"]
print("Skater breakouts (beat the projection by the most WAR):")
print(top(sk, 10, True).to_string())
print("\nSkater collapses:")
print(top(sk, 10, False).to_string())

Skater breakouts (beat the projection by the most WAR):
               name   season   age group       toi  proj_war_60  actual_war_60  residual_war
0     Evgeni Malkin  2011-12  25.0     F  1575.117        0.070          0.260         4.998
1     Alex Ovechkin  2009-10  24.0     F  1569.417        0.111          0.263         3.978
2     Sidney Crosby  2018-19  31.0     F  1658.017        0.094          0.236         3.927
3    Filip Forsberg  2021-22  27.0     F  1248.750        0.087          0.267         3.749
4     Erik Karlsson  2022-23  32.0     D  2100.433        0.026          0.132         3.695
5        Zach Hyman  2023-24  31.0     F  1560.033        0.069          0.207         3.592
6  William Karlsson  2023-24  31.0     F  1232.117        0.025          0.197         3.536
7     Cole Caufield  2025-26  25.0     F  1472.867        0.112          0.254         3.484
8    Steven Stamkos  2018-19  28.0     F  1499.567        0.098          0.233         3.369
9   Johnny Gau

In [4]:
print("Goalie breakouts:")
print(top(gk, 10, True).to_string())
print("\nGoalie collapses:")
print(top(gk, 10, False).to_string())

Goalie breakouts:
                name   season   age group       toi  proj_war_60  actual_war_60  residual_war
0         Tim Thomas  2010-11  36.0     G  3363.967        0.049          0.167         6.624
1      Linus Ullmark  2022-23  29.0     G  2882.200        0.052          0.182         6.229
2         Mike Smith  2011-12  29.0     G  3903.200        0.037          0.129         5.987
3    Igor Shesterkin  2021-22  26.0     G  3070.533        0.056          0.165         5.575
4        Juuse Saros  2022-23  27.0     G  3809.983        0.073          0.157         5.298
5   Sergei Bobrovsky  2016-17  28.0     G  3707.067        0.041          0.124         5.110
6        Pekka Rinne  2017-18  35.0     G  3475.450        0.025          0.111         4.951
7       Ilya Sorokin  2022-23  27.0     G  3587.067        0.062          0.142         4.780
8       Antti Raanta  2017-18  28.0     G  2599.050        0.059          0.169         4.780
9  Frederik Andersen  2021-22  32.0     G 

## 2. Luck vs. real change

A miss has two kinds of causes:

- **Sampling luck:** a few bounces, a hot or cold shooting stretch. Over more minutes it averages out, so its variance shrinks like **b / minutes**.
- **Everything else:** a new role, new linemates, an injury played through, or a real jump or decline. More minutes don't make it smaller, so it's a constant **a**.

So the variance of a miss should look like **a + b / minutes**. We fit a and b for each group (minutes-weighted least squares of miss² on 1 / minutes) and check the fit against the actual variance in six minutes bands.

One caution: **a** isn't pure talent change. It also holds the projection's own uncertainty about how good the player really was, which more minutes *next* season can't fix.

In [5]:
fit = luck_fit(res)
print("Fit: variance of a miss (WAR/60 squared) = a + b / minutes")
print(fit.rename(index=NAMES).to_string(float_format=lambda v: f"{v:.6f}"))

rows = []
for g, d in res.groupby("group"):
    d = d.assign(band=pd.qcut(d["toi"], 6))
    for band, e in d.groupby("band", observed=True):
        rows.append({"group": NAMES[g], "minutes": f"{band.left:,.0f} to {band.right:,.0f}", "player-seasons": len(e),
                     "actual variance": np.average(e["residual"] ** 2, weights=e["toi"]),
                     "fitted variance": np.average(fit.loc[g, "a"] + fit.loc[g, "b"] / e["toi"], weights=e["toi"])})
print("\nDoes the fit match? Variance by minutes band:")
pd.DataFrame(rows).set_index(["group", "minutes"]).round(5)

Fit: variance of a miss (WAR/60 squared) = a + b / minutes
                  a        b
Forwards   0.001006 1.733648
Defensemen 0.000336 0.713222
Goalies    0.000245 5.043218

Does the fit match? Variance by minutes band:


player-seasons  actual variance  fitted variance
group      minutes                                                         
Defensemen 301 to 710                 507          0.00179          0.00174
           710 to 1,066               506          0.00112          0.00113
           1,066 to 1,331             507          0.00090          0.00093
           1,331 to 1,546             506          0.00078          0.00083
           1,546 to 1,753             506          0.00075          0.00077
           1,753 to 2,411             507          0.00078          0.00071
Forwards   300 to 641                 921          0.00492          0.00467
           641 to 886                 920          0.00305          0.00325
           886 to 1,088               921          0.00254          0.00276
           1,088 to 1,257             920          0.00244          0.00248
           1,257 to 1,425             920          0.00234          0.00230
           1,425 to 1,885             921          0.00228          0.00212
Goalies    1,005 to 1,408             131          0.00435          0.00438
           1,408 to 1,793             130          0.00327          0.00339
           1,793 to 2,324             130          0.00262          0.00268
           2,324 to 2,942             130          0.00233          0.00215
           2,942 to 3,418             129          0.00185          0.00182
           3,418 to 4,459             131          0.00152          0.00159

In [6]:
for g, mins in [("F", [800, 1400]), ("D", [800, 1400]), ("G", [2000, 3000])]:
    for m in mins:
        sd = expected_sd(fit, [g], [m])[0]
        print(f"{ONE[g]} at {m:,} minutes: about {luck_share(fit, g, m):.0%} of a surprise season is luck. "
              f"A typical miss is {sd:.3f} WAR/60, or {sd * m / 60:.2f} WAR over the season.")

A forward at 800 minutes: about 68% of a surprise season is luck. A typical miss is 0.056 WAR/60, or 0.75 WAR over the season.
A forward at 1,400 minutes: about 55% of a surprise season is luck. A typical miss is 0.047 WAR/60, or 1.11 WAR over the season.
A defenseman at 800 minutes: about 73% of a surprise season is luck. A typical miss is 0.035 WAR/60, or 0.47 WAR over the season.
A defenseman at 1,400 minutes: about 60% of a surprise season is luck. A typical miss is 0.029 WAR/60, or 0.68 WAR over the season.
A goalie at 2,000 minutes: about 91% of a surprise season is luck. A typical miss is 0.053 WAR/60, or 1.75 WAR over the season.
A goalie at 3,000 minutes: about 87% of a surprise season is luck. A typical miss is 0.044 WAR/60, or 2.19 WAR over the season.


## 3. Does a surprise stick?

If a player beats his projection this season, does he beat his (new) projection again next season? For players tested in back-to-back target seasons, we measure the minutes-weighted correlation and slope between the two misses (each pair weighted by its shorter season). Misses are centered first.

**Important:** next season's projection already includes this season. With 3 equal-weight seasons and 1,500 regression minutes, this season carries roughly a quarter of next year's projection. So this number is the **extra** share that sticks beyond what the blend already carries. Near zero means the blend is carrying about the right amount.

In [7]:
res = add_z(res, fit)               # adds the scaled miss z and centered versions (residual_c, z_c)
pers = persistence(res)
print("Minutes-weighted correlation and slope between a miss and the same player's miss next season:")
pers.assign(group=pers["group"].map(NAMES)).round(3)

Minutes-weighted correlation and slope between a miss and the same player's miss next season:


,group,age_bucket,pairs,corr,slope
0,Defensemen,all,2020,0.056,0.056
1,Defensemen,<=21,85,-0.048,-0.045
2,Defensemen,22-23,274,0.122,0.123
3,Defensemen,24-26,565,0.015,0.016
4,Defensemen,27-29,483,0.098,0.095
5,Defensemen,30-32,363,0.023,0.025
6,Defensemen,33+,250,-0.007,-0.007
7,Forwards,all,3635,0.034,0.033
8,Forwards,<=21,242,0.062,0.060
9,Forwards,22-23,520,0.071,0.065


In [8]:
# What carries into next year's true talent in the simulator: the overall slope per group.
# A negative slope is noise (a surprise can't make next year move the other way on average), so it counts as 0.
carry = {r.group: max(0.0, r.slope) for r in pers[pers["age_bucket"] == "all"].itertuples()}
print("Persistent share used in the simulator:", {NAMES[g]: round(v, 3) for g, v in carry.items()})

Persistent share used in the simulator: {'Defensemen': 0.056, 'Forwards': 0.033, 'Goalies': 0.0}


**Read it:** the extra stickiness is small. Most of what isn't luck is **one-year change**: a role or linemate change, an injury, a hot streak that isn't just shooting luck. If the non-luck part (a) were mostly permanent, next season's miss would correlate with this one at about 0.3, because the blend only carries a quarter of it. It's far below that.

## 4. Misses by age, experience and draft position

For each bucket: player-seasons, the raw and centered average miss, the spread (SD), percentiles of the centered miss, and skew (positive = a longer upside tail). All minutes-weighted, WAR per 60. Buckets with fewer than 50 player-seasons are merged with their smaller neighbour (printed below each table). Goalies get age buckets only: their samples are small.

The noise checks:
- **mean_flag:** the bucket's centered average is more than 2 standard errors from 0, its group's average.
- **sd_flag:** its spread (in scaled units, z) is outside the range you'd get 90% of the time from a random set of the same size from its group (1,000 random draws).

In [9]:
COLS = ["group", "bucket", "n", "mean_raw", "mean", "se", "mean_flag", "sd", "z_sd", "sd_flag",
        "p10", "p25", "p50", "p75", "p90", "skew"]

age_sk, age_maps, notes = bucket_stats(res, "age_bucket", AGE_ORDER, groups=("F", "D"))
age_g, age_maps_g, notes_g = bucket_stats(res, "age_bucket", AGE_ORDER, groups=("G",))
age_maps.update(age_maps_g)
age_tbl = pd.concat([age_sk, age_g], ignore_index=True)
print("By age:")
print(age_tbl[COLS].round(4).to_string())
print("Merges:", "; ".join(notes + notes_g) or "none")

By age:
   group        bucket     n  mean_raw    mean      se  mean_flag      sd    z_sd  sd_flag     p10     p25     p50     p75     p90    skew
0      F          <=21   340    0.0096  0.0145  0.0031       True  0.0539  1.0666    False -0.0601 -0.0228  0.0168  0.0507  0.0800 -0.1234
1      F         22-23   740   -0.0020  0.0030  0.0022      False  0.0551  1.0527     True -0.0674 -0.0352  0.0025  0.0401  0.0747  0.0298
2      F         24-26  1470   -0.0046  0.0004  0.0014      False  0.0517  1.0020    False -0.0660 -0.0328  0.0023  0.0329  0.0618  0.1050
3      F         27-29  1299   -0.0064 -0.0014  0.0015      False  0.0506  0.9921    False -0.0650 -0.0358 -0.0023  0.0318  0.0649  0.1136
4      F         30-32   929   -0.0072 -0.0022  0.0017      False  0.0492  0.9757    False -0.0631 -0.0352 -0.0029  0.0301  0.0607  0.1881
5      F           33+   745   -0.0099 -0.0050  0.0018       True  0.0478  0.9181     True -0.0639 -0.0351 -0.0064  0.0265  0.0592  0.0797
6      D          <

In [10]:
exp_tbl, exp_maps, notes = bucket_stats(res, "exp_bucket", EXP_ORDER, groups=("F", "D"))
print("By NHL experience (prior seasons with 200+ minutes):")
print(exp_tbl[COLS].round(4).to_string())
print("Merges:", "; ".join(notes) or "none")

By NHL experience (prior seasons with 200+ minutes):
  group bucket     n  mean_raw    mean      se  mean_flag      sd    z_sd  sd_flag     p10     p25     p50     p75     p90    skew
0     F    0-1   950   -0.0074 -0.0025  0.0020      False  0.0570  1.0369    False -0.0757 -0.0421 -0.0031  0.0381  0.0685  0.0301
1     F      2   589   -0.0023  0.0027  0.0023      False  0.0539  1.0460    False -0.0648 -0.0339  0.0025  0.0389  0.0728  0.1410
2     F      3   524   -0.0062 -0.0012  0.0024      False  0.0512  0.9825    False -0.0670 -0.0366  0.0003  0.0327  0.0640  0.0128
3     F    4-6  1377   -0.0052 -0.0002  0.0014      False  0.0495  0.9764    False -0.0629 -0.0334 -0.0001  0.0322  0.0621  0.1596
4     F     7+  2083   -0.0044  0.0006  0.0011      False  0.0497  0.9917    False -0.0623 -0.0322 -0.0009  0.0326  0.0650  0.1297
5     D    0-1   518   -0.0018  0.0019  0.0016      False  0.0331  1.0249    False -0.0384 -0.0205  0.0002  0.0206  0.0458  0.0756
6     D      2   329   -0.0006

In [11]:
draft_tbl, draft_maps, notes = bucket_stats(res, "draft_bucket", DRAFT_ORDER, groups=("F", "D"))
print("By draft position:")
print(draft_tbl[COLS].round(4).to_string())
print("Merges:", "; ".join(notes) or "none")

By draft position:
   group     bucket     n  mean_raw    mean      se  mean_flag      sd    z_sd  sd_flag     p10     p25     p50     p75     p90    skew
0      F        1-5   665    0.0074  0.0124  0.0021       True  0.0517  1.0778     True -0.0557 -0.0227  0.0122  0.0450  0.0795  0.1209
1      F       6-15   793   -0.0019  0.0031  0.0018      False  0.0496  0.9870    False -0.0611 -0.0312  0.0042  0.0353  0.0657  0.0434
2      F      16-32   952   -0.0042  0.0007  0.0017      False  0.0510  0.9952    False -0.0650 -0.0308 -0.0005  0.0344  0.0644  0.0980
3      F      33-64   849   -0.0080 -0.0030  0.0018      False  0.0507  0.9732    False -0.0652 -0.0351 -0.0050  0.0277  0.0634  0.1421
4      F        65+  1672   -0.0083 -0.0034  0.0013       True  0.0509  0.9652     True -0.0674 -0.0383 -0.0042  0.0299  0.0615  0.1055
5      F  undrafted   592   -0.0170 -0.0120  0.0023       True  0.0517  0.9529    False -0.0748 -0.0451 -0.0157  0.0220  0.0549  0.1461
6      D        1-5   256   -

## 5. What actually moves the range?

A summary of the flags above, then the question from notebook 16: top picks beat their NHLe as rookies. **Do they keep beating their projection once they have NHL history?** For skaters 23 and under, an age × draft table (forwards and defensemen together, each centered within his group).

In [12]:
flags = pd.concat([age_tbl.assign(by="age"), exp_tbl.assign(by="experience"), draft_tbl.assign(by="draft")])
moved = flags[flags["mean_flag"] | flags["sd_flag"]][["by", "group", "bucket", "n", "mean", "se", "mean_flag", "z_sd", "sd_flag"]]
print("Buckets whose average or spread differs from their group beyond noise:")
moved.round(4).reset_index(drop=True)

Buckets whose average or spread differs from their group beyond noise:


,by,group,bucket,n,mean,se,mean_flag,z_sd,sd_flag
0,age,F,<=21,340,0.0145,0.0031,True,1.0666,False
1,age,F,22-23,740,0.0030,0.0022,False,1.0527,True
2,age,F,33+,745,-0.0050,0.0018,True,0.9181,True
3,age,D,<=21,122,0.0053,0.0033,False,1.1627,True
4,age,D,22-23,377,0.0042,0.0018,True,1.0377,False
5,age,D,33+,460,-0.0051,0.0014,True,0.9416,False
6,draft,F,1-5,665,0.0124,0.0021,True,1.0778,True
7,draft,F,65+,1672,-0.0034,0.0013,True,0.9652,True
8,draft,F,undrafted,592,-0.0120,0.0023,True,0.9529,False
9,draft,D,6-15,427,0.0032,0.0015,True,1.0362,False


In [13]:
young = res[(res["group"] != "G") & (res["age"] <= 23)].copy()
young["age_x_draft"] = young["age_bucket"] + " | " + young["draft_bucket"]

def cell(d):
    w = d["toi"].to_numpy()
    n_eff = w.sum() ** 2 / (w ** 2).sum()
    m, se = wmean(d["residual_c"], w), wsd(d["residual_c"].to_numpy(), w) / np.sqrt(n_eff)
    return pd.Series({"n": len(d), "mean": m, "se": se, "flag": abs(m) > 2 * se,
                      "beat projection": np.average(d["residual_c"] > 0, weights=w)})

axd = young.groupby(["age_bucket", "draft_bucket"]).apply(cell, include_groups=False)
axd = axd.reindex(pd.MultiIndex.from_product([["<=21", "22-23"], DRAFT_ORDER])).dropna(how="all")
print("Skaters 23 and under: centered miss (WAR/60) by age and draft position")
axd.round(4)

Skaters 23 and under: centered miss (WAR/60) by age and draft position


n    mean      se   flag  beat projection
<=21  1-5        133.0  0.0223  0.0046   True           0.7201
      6-15       144.0  0.0065  0.0039  False           0.5714
      16-32      102.0  0.0051  0.0055  False           0.4689
      33-64       53.0  0.0071  0.0067  False           0.5088
      65+         30.0  0.0096  0.0100  False           0.5431
22-23 1-5        133.0  0.0090  0.0043   True           0.5757
      6-15       200.0  0.0056  0.0034  False           0.5293
      16-32      232.0  0.0039  0.0032  False           0.5315
      33-64      235.0  0.0002  0.0036  False           0.5256
      65+        277.0  0.0005  0.0031  False           0.4863
      undrafted   40.0 -0.0013  0.0065  False           0.5896

## 6. Does pre-NHL scoring help with young players?

For skaters 23 and under with 3 or fewer prior NHL seasons: take their **best pre-NHL season** (before their first NHL season with 20+ games, 20+ games in that league) converted with notebook 16's chosen factors (forward/defense factor where there is one, times the age multiplier). Does a higher NHLe mean a player beats his projection, even after his NHL seasons are in the projection? Then: does it add anything once age and draft position are known?

In [14]:
nhle = best_nhle(careers, factors, multipliers)
yn = res[(res["group"] != "G") & (res["age"] <= 23) & (res["experience"] <= 3)].merge(nhle, on="playerId", how="left")
has = yn["nhle_ppg"].notna()
print(f"Young skater-seasons: {len(yn):,}; with a pre-NHL NHLe: {has.sum():,} ({has.mean():.0%})")
yn = yn[has].copy()
r, slope = wcorr(yn["nhle_ppg"], yn["residual_c"], yn["toi"])
print(f"Minutes-weighted correlation between best NHLe and the miss: {r:.3f}")

yn["NHLe group"] = pd.qcut(yn["nhle_ppg"], 5, labels=["lowest", "low", "middle", "high", "highest"])
binned = yn.groupby("NHLe group", observed=True).apply(lambda d: pd.Series({
    "n": len(d), "NHLe (NHL points/game)": d["nhle_ppg"].mean(), "average miss": wmean(d["residual_c"], d["toi"])}),
    include_groups=False)
binned.round(4)

Young skater-seasons: 1,474; with a pre-NHL NHLe: 1,474 (100%)
Minutes-weighted correlation between best NHLe and the miss: 0.185


,n,NHLe (NHL points/game),average miss
NHLe group,,,
lowest,296.0,0.2319,-0.0019
low,294.0,0.3449,-0.0016
middle,294.0,0.4133,0.0018
high,297.0,0.4839,0.0066
highest,293.0,0.6055,0.0213


In [15]:
# Weighted regression of the miss on age bucket and draft bucket (as yes/no columns), with and without NHLe
X0 = pd.get_dummies(yn[["age_bucket", "draft_bucket"]], drop_first=True).astype(float)
X0.insert(0, "constant", 1.0)
X1 = X0.assign(nhle_ppg=yn["nhle_ppg"].to_numpy())
b0, se0, r2_0 = wls(yn["residual_c"], X0, yn["toi"])
b1, se1, r2_1 = wls(yn["residual_c"], X1, yn["toi"])
coef, se = b1[-1], se1[-1]
print(f"Age + draft only:        R^2 = {r2_0:.4f}")
print(f"Age + draft + NHLe:      R^2 = {r2_1:.4f}")
print(f"NHLe coefficient: {coef:+.4f} WAR/60 per NHL point per game (standard error {se:.4f}, {coef / se:+.1f} standard errors)")
verdict = ("adds something beyond age and draft" if abs(coef / se) > 2
           else "adds nothing reliable beyond age and draft")
print(f"Verdict: NHLe {verdict}.")
pd.DataFrame({"coefficient": b1, "standard error": se1}, index=X1.columns).round(4)

Age + draft only:        R^2 = 0.0158
Age + draft + NHLe:      R^2 = 0.0413
NHLe coefficient: +0.0584 WAR/60 per NHL point per game (standard error 0.0094, +6.2 standard errors)
Verdict: NHLe adds something beyond age and draft.


,coefficient,standard error
constant,-0.0146,0.0056
age_bucket_<=21,0.0046,0.0029
draft_bucket_16-32,-0.0073,0.0041
draft_bucket_33-64,-0.0088,0.0043
draft_bucket_6-15,-0.0061,0.0039
draft_bucket_65+,-0.0074,0.0045
draft_bucket_undrafted,-0.0075,0.0095
nhle_ppg,0.0584,0.0094


## 7. Dropouts: the misses we never see

A miss needs a next season. A regular (500+ minutes) who plays under 200 minutes (skaters) or 500 (goalies) the next season, or not at all, never shows up in the table. Cutoffs are prorated for short seasons. This mixes very different stories: **injuries, demotions to the AHL, moves to Europe and retirements**. Because these players are missing from the misses, **the downside of the ranges is understated** (survivor bias), especially for older players.

In [16]:
drop = dropouts(history, bios)
print(f"{len(drop):,} regular seasons (500+ minutes) with a following season in the data")
by_age = drop.groupby(["group", "age_bucket"])["dropout"].agg(["size", "mean"]).rename(columns={"size": "regulars", "mean": "dropout rate"})
by_age = by_age.unstack(0).reindex(AGE_ORDER)
by_age.round(3)

11,155 regular seasons (500+ minutes) with a following season in the data


regulars            dropout rate              
group             D     F    G            D      F      G
age_bucket                                               
<=21            237   525   21        0.076  0.065  0.190
22-23           494   865  106        0.091  0.057  0.179
24-26           941  1676  274        0.084  0.081  0.117
27-29           826  1463  300        0.091  0.090  0.160
30-32           608  1002  227        0.123  0.108  0.132
33+             529   827  234        0.267  0.249  0.291

In [17]:
by_season = drop.groupby(["season", "group"])["dropout"].mean().unstack().reindex(columns=["F", "D", "G"])
by_season.index = [f"{label(s)} -> {label(s + 1)}" for s in by_season.index]
print("Dropout rate by season (season t -> t+1; short seasons are prorated):")
by_season.round(3)

Dropout rate by season (season t -> t+1; short seasons are prorated):


group,F,D,G
2008-09 -> 2009-10,0.107,0.146,0.188
2009-10 -> 2010-11,0.127,0.142,0.161
2010-11 -> 2011-12,0.085,0.081,0.154
2011-12 -> 2012-13,0.104,0.134,0.159
2012-13 -> 2013-14,0.120,0.142,0.180
2013-14 -> 2014-15,0.092,0.120,0.243
2014-15 -> 2015-16,0.111,0.151,0.138
2015-16 -> 2016-17,0.128,0.112,0.188
2016-17 -> 2017-18,0.120,0.089,0.111
2017-18 -> 2018-19,0.111,0.100,0.214


## 8. Calibration: do the ranges hold up on later seasons?

The real test. Build the ranges on target seasons **through 2017-18** only (the luck fit, the group averages and the age buckets too), then check the later seasons **from 2018-19 on**: what share of actual misses lands inside each bucket's range?

| Range | Should contain |
|---|---|
| 5th to 95th percentile | 90% |
| 10th to 90th | 80% |
| 25th to 75th | 50% |

Four versions:
- **raw:** percentiles of the miss in WAR/60, the same width for everyone in a bucket.
- **scaled:** percentiles of z = miss / its expected size at the player's minutes (from the luck fit), converted back at his minutes. A 400-minute player gets a wider range than a 1,600-minute player, because more of his season is luck.
- Each **uncentered** (the projection's bias kept) and **centered** (bias removed, as the simulator will draw).

In [18]:
cal = calibrate(res)
score = calibration_score(cal)
print(f"Test player-seasons (2018-19 on): {len(cal):,}")
print("Average distance from the target share (lower is better):")
print(score.round(3).to_string())
best = score.idxmin()
print(f"Best calibrated: {best}")
cov = coverage(cal.assign(group=cal["group"].map(NAMES)), "group")
cov.round(3)

Test player-seasons (2018-19 on): 4,188
Average distance from the target share (lower is better):
raw                 0.049
raw, centered       0.051
scaled              0.023
scaled, centered    0.027
Best calibrated: scaled


version                raw                     raw, centered                        scaled                     scaled, centered                    
range          n      5-95     10-90     25-75          5-95     10-90     25-75      5-95     10-90     25-75             5-95     10-90     25-75
group                                                                                                                                              
Defensemen  1348  0.859792  0.739614  0.460682      0.860534  0.742582  0.452522  0.883531  0.762611  0.494065         0.876113  0.767062  0.481454
Forwards    2470  0.863563  0.749798  0.467611       0.85749  0.745344  0.462753  0.873684  0.780162  0.478138         0.868016  0.764777  0.478543
Goalies      370  0.848649  0.743243   0.42973      0.854054  0.735135  0.432432  0.881081  0.783784  0.454054         0.883784  0.783784  0.451351

In [19]:
print("By age bucket (all groups):")
coverage(cal, "age_bucket").reindex(AGE_ORDER).round(3)

By age bucket (all groups):


version                raw                     raw, centered                        scaled                     scaled, centered                    
range          n      5-95     10-90     25-75          5-95     10-90     25-75      5-95     10-90     25-75             5-95     10-90     25-75
age_bucket                                                                                                                                         
<=21         178   0.88764  0.797753  0.589888      0.876404  0.792135  0.539326  0.926966  0.842697  0.595506         0.910112  0.831461  0.578652
22-23        484  0.865702  0.733471  0.464876      0.865702  0.714876  0.440083  0.884298  0.762397  0.471074         0.863636  0.737603  0.448347
24-26       1079  0.852641   0.73772  0.430955      0.848007  0.741427  0.439296  0.868397  0.777572  0.459685         0.865616  0.765524  0.467099
27-29       1065  0.880751  0.752113  0.461972      0.877934   0.74554  0.465728  0.892019  0.774648  0.489202         0.886385  0.768075  0.489202
30-32        788  0.852792  0.739848  0.467005      0.846447  0.747462  0.455584  0.871827  0.765228  0.479695         0.865482  0.766497  0.467005
33+          594  0.840067  0.752525   0.47138      0.845118  0.747475  0.462963  0.855219  0.772727  0.481481         0.861953  0.774411  0.479798

In [20]:
cal["minutes band"] = np.select(
    [(cal["group"] != "G") & (cal["toi"] < 800), (cal["group"] != "G") & (cal["toi"] < 1300), cal["group"] != "G",
     cal["toi"] < 2000, cal["toi"] < 3000],
    ["skaters: under 800", "skaters: 800-1,300", "skaters: 1,300+", "goalies: under 2,000", "goalies: 2,000-3,000"],
    "goalies: 3,000+")
print("By minutes band:")
coverage(cal, "minutes band").round(3)

By minutes band:


version                          raw                     raw, centered                        scaled                     scaled, centered                    
range                    n      5-95     10-90     25-75          5-95     10-90     25-75      5-95     10-90     25-75             5-95     10-90     25-75
minutes band                                                                                                                                                 
goalies: 2,000-3,000   119  0.865546  0.789916  0.453782      0.865546  0.781513  0.453782  0.840336  0.815126  0.453782         0.840336  0.806723  0.453782
goalies: 3,000+         93  0.913978  0.795699   0.44086      0.924731  0.784946  0.451613  0.913978  0.731183  0.387097         0.913978   0.72043  0.397849
goalies: under 2,000   158  0.797468  0.677215  0.405063      0.803797  0.670886  0.405063  0.892405  0.791139  0.493671         0.898734  0.803797  0.481013
skaters: 1,300+       1434  0.889819  0.771967  0.483264      0.892608  0.786611  0.488842  0.859135  0.735704  0.453278         0.862622  0.746862  0.454672
skaters: 800-1,300    1465  0.885324  0.775427  0.494198      0.885324  0.770648  0.484642  0.891468       0.8  0.503754         0.883959  0.780205  0.490102
skaters: under 800     919  0.782372  0.659412  0.390642      0.762786  0.636561  0.372144  0.882481  0.792165  0.499456         0.862894  0.771491  0.501632

## 9. Simulation recipe (design) and a small demo

**How the simulator will draw a player's season:**

1. Start from his projection (notebook 15) and his expected minutes.
2. Pick his bucket: his group's (merged) age bucket. For skaters 23 and under, use his age × draft cell instead (forwards and defensemen pooled, as tested), but only when that cell is flagged in section 5 and has 50+ player-seasons.
3. Draw one **scaled, centered** miss z from that bucket's history (an actual past miss, not a bell curve, so real skew and fat tails come through; each past season's chance is in proportion to its minutes).
4. Convert it at his minutes: miss = z × sqrt(a + b / minutes).
5. Split it into three parts, each its average share of the miss:
   - **sampling luck** = (b / minutes) / (a + b / minutes) × miss. This season only.
   - **persistent share** = the slope from section 3 × miss. Carries into next year's true talent, on top of what the blend already carries when the simulator re-blends the simulated season.
   - **one-year change** = the rest. This season only.

**Note for Phase 2:** the game simulator will add its own game-to-game randomness, so it may use only the non-luck parts (one-year change and persistent share) to avoid counting luck twice. That decision is made in Phase 2.

**Demo:** 1,000 draws of 2026-27 WAR at 2025-26 minutes for three skaters, picked by rule from `data/true_talent_2026-27.csv`: the top-5 pick aged 21 or under with the most 2025-26 minutes, the 27-year-old with the most minutes, and the 34+ skater with the most minutes.

In [21]:
# Each player-season's draw bucket: his group's merged age bucket ...
res["sim_bucket"] = [age_maps[g][a] for g, a in zip(res["group"], res["age_bucket"])]
# ... or, for skaters 23 and under, his age x draft cell when it's flagged and has 50+ player-seasons
use_cells = [f"{a} | {d}" for (a, d), r in axd.iterrows() if r["flag"] and r["n"] >= 50]
young_cell = (res["group"] != "G") & (res["age"] <= 23)
key = res["age_bucket"] + " | " + res["draft_bucket"]
in_cell = young_cell & key.isin(use_cells)
res.loc[in_cell, "sim_bucket"] = key
# Those cells pool forwards and defensemen (the cell was tested on both together; z has no units)
res["pool_group"] = np.where(in_cell, "F+D", res["group"])
print("Age x draft cells with their own draw bucket (forwards and defensemen pooled):", ", ".join(use_cells) or "none")
print(res[in_cell].groupby("sim_bucket").size().to_string())
pools = draw_pools(res, "sim_bucket", "pool_group")
print(f"{len(pools)} draw buckets")

Age x draft cells with their own draw bucket (forwards and defensemen pooled): <=21 | 1-5, 22-23 | 1-5
sim_bucket
22-23 | 1-5    133
<=21 | 1-5     133
19 draw buckets


In [22]:
tt = pd.read_csv(DATA / "true_talent_2026-27.csv").merge(bios[["playerId", "draft_overall"]], on="playerId", how="left")
mins_2025 = history[history["season"] == 2025].groupby("playerId")["toi"].sum()
tt["minutes"] = tt["playerId"].map(mins_2025)
tt = tt[(tt["position"] != "G") & tt["minutes"].notna()].copy()
tt["group"] = np.where(tt["position"] == "D", "D", "F")

picks = [
    ("top-5 pick, 21 or under", tt[(tt["draft_overall"] <= 5) & (tt["age_next"] <= 21)]),
    ("27-year-old regular", tt[tt["age_next"] == 27]),
    ("veteran, 34+", tt[tt["age_next"] >= 34]),
]
players = pd.concat([d.nlargest(1, "minutes").assign(role=role) for role, d in picks], ignore_index=True)

def pool_key(r):
    a = age_bucket([r["age_next"]])[0]
    cell_key = f"{a} | {draft_bucket([r['draft_overall']])[0]}"
    return cell_key if (r["age_next"] <= 23 and cell_key in use_cells) else age_maps[r["group"]][a]

players["pool_key"] = players.apply(pool_key, axis=1)
players["pool_group"] = np.where(players["pool_key"].str.contains("|", regex=False), "F+D", players["group"])
players[["role", "name", "position", "age_next", "draft_overall", "minutes", "proj_war_60", "pool_group", "pool_key"]].round(3)

,role,name,position,age_next,draft_overall,minutes,proj_war_60,pool_group,pool_key
0,"top-5 pick, 21 or under",Matthew Schaefer,D,19.0,1.0,2023.983,0.064,F+D,<=21 | 1-5
1,27-year-old regular,Quinn Hughes,D,27.0,7.0,2052.583,0.080,D,27-29
2,"veteran, 34+",Erik Karlsson,D,36.0,15.0,1770.033,0.037,D,33+


In [23]:
draws = draw(players[["playerId", "group", "pool_group", "pool_key", "minutes", "proj_war_60"]], pools, fit, carry, n=1000)
summary = draws.groupby("playerId").apply(lambda d: pd.Series({
    "projected WAR": None, "10th": d["war"].quantile(0.10), "50th": d["war"].quantile(0.50),
    "90th": d["war"].quantile(0.90), "upside (90th - 50th)": d["war"].quantile(0.9) - d["war"].quantile(0.5),
    "downside (50th - 10th)": d["war"].quantile(0.5) - d["war"].quantile(0.1),
    "luck SD (WAR/60)": d["luck"].std(), "one-year SD": d["one_year"].std(), "persistent SD": d["persistent"].std()}),
    include_groups=False)
summary["projected WAR"] = players.set_index("playerId")["proj_war_60"] * players.set_index("playerId")["minutes"] / 60
summary["90th above projection"] = summary["90th"] - summary["projected WAR"]
summary["10th below projection"] = summary["projected WAR"] - summary["10th"]
summary.index = summary.index.map(players.set_index("playerId")["name"])
summary.astype(float).round(3)

,projected WAR,10th,50th,90th,upside (90th - 50th),downside (50th - 10th),luck SD (WAR/60),one-year SD,persistent SD,90th above projection,10th below projection
playerId,,,,,,,,,,,
Erik Karlsson,1.095,0.004,0.942,1.955,1.013,0.938,0.014,0.010,0.001,0.860,1.091
Quinn Hughes,2.748,1.644,2.714,3.886,1.172,1.070,0.013,0.011,0.001,1.138,1.104
Matthew Schaefer,2.151,1.040,2.623,3.834,1.211,1.583,0.016,0.013,0.002,1.683,1.110


In [24]:
# A small text histogram of each player's 1,000 simulated 2026-27 WAR totals
edges = np.arange(np.floor(draws["war"].min() * 2) / 2, draws["war"].max() + 0.5, 0.5)
for r in players.itertuples():
    w = draws.loc[draws["playerId"] == r.playerId, "war"]
    counts, _ = np.histogram(w, bins=edges)
    print(f"\n{r.name} ({r.role}), WAR at {r.minutes:,.0f} minutes:")
    for lo, c in zip(edges[:-1], counts):
        if c:
            print(f"  {lo:5.1f} to {lo + 0.5:4.1f} | {'#' * int(round(c / 10))} {c}")


Matthew Schaefer (top-5 pick, 21 or under), WAR at 2,024 minutes:
   -0.5 to  0.0 |  1
    0.0 to  0.5 | ## 15
    0.5 to  1.0 | ######## 81
    1.0 to  1.5 | ######## 75
    1.5 to  2.0 | ######### 90
    2.0 to  2.5 | #################### 197
    2.5 to  3.0 | ################### 190
    3.0 to  3.5 | ################## 185
    3.5 to  4.0 | ########## 97
    4.0 to  4.5 | #### 42
    4.5 to  5.0 | ## 23
    5.0 to  5.5 |  4

Quinn Hughes (27-year-old regular), WAR at 2,053 minutes:
    0.0 to  0.5 |  4
    0.5 to  1.0 | # 14
    1.0 to  1.5 | ##### 46
    1.5 to  2.0 | ################# 168
    2.0 to  2.5 | ################# 173
    2.5 to  3.0 | #################### 205
    3.0 to  3.5 | #################### 205
    3.5 to  4.0 | ########## 100
    4.0 to  4.5 | ##### 54
    4.5 to  5.0 | ## 23
    5.0 to  5.5 | # 6
    5.5 to  6.0 |  1
    6.0 to  6.5 |  1

Erik Karlsson (veteran, 34+), WAR at 1,770 minutes:
   -1.5 to -1.0 |  4
   -1.0 to -0.5 | ## 15
   -0.5 to  0.0 | ########

In [25]:
s = summary.astype(float)
young_name, vet_name = players.loc[0, "name"], players.loc[2, "name"]
print("Hockey check, measured from each player's projection (WAR):")
for n in [young_name, players.loc[1, "name"], vet_name]:
    up, down = s.loc[n, "90th above projection"], s.loc[n, "10th below projection"]
    lean = "more upside" if up > down + 0.1 else "more downside" if down > up + 0.1 else "balanced"
    print(f"  {n:18s} 90th is {up:+.2f} above, 10th is {down:.2f} below, median {s.loc[n, '50th'] - s.loc[n, 'projected WAR']:+.2f}: {lean}")

Hockey check, measured from each player's projection (WAR):
  Matthew Schaefer   90th is +1.68 above, 10th is 1.11 below, median +0.47: more upside
  Quinn Hughes       90th is +1.14 above, 10th is 1.10 below, median -0.03: balanced
  Erik Karlsson      90th is +0.86 above, 10th is 1.09 below, median -0.15: more downside


## 10. Save

- `dev_residuals.csv`: every tested player-season with its miss, scaled miss, centered versions and draw bucket.
- `dev_ranges.csv`: one tidy table with a `kind` column: bucket rows (age, experience, draft, age x draft) with percentiles of the centered miss and of z, the luck fit (a, b), the persistence rows, and each group's average miss (the bias that centering removes).

In [26]:
res.to_csv(OUT / "dev_residuals.csv", index=False)

axd_rows = axd.reset_index().rename(columns={"level_0": "age_bucket", "level_1": "draft_bucket"})
axd_rows = axd_rows.assign(bucket=axd_rows["age_bucket"] + " | " + axd_rows["draft_bucket"], group="F+D")
ranges = pd.concat([
    age_tbl.assign(kind="age"), exp_tbl.assign(kind="experience"), draft_tbl.assign(kind="draft"),
    axd_rows[["group", "bucket", "n", "mean", "se", "flag"]].rename(columns={"flag": "mean_flag"}).assign(kind="age_x_draft"),
    fit.reset_index().rename(columns={"index": "group"}).assign(kind="luck_fit", bucket="all"),
    pers.rename(columns={"age_bucket": "bucket", "pairs": "n"}).assign(kind="persistence",
                                                                      carry=lambda d: d["group"].map(carry).where(d["bucket"] == "all")),
    group_means(res).reset_index().rename(columns={"residual": "mean_raw", "z": "z_mean_raw"}).assign(kind="group_mean", bucket="all"),
], ignore_index=True)
front = ["kind", "group", "bucket", "n"]
ranges = ranges[front + [c for c in ranges.columns if c not in front]]
ranges.to_csv(OUT / "dev_ranges.csv", index=False)
print(f"Saved dev_residuals.csv ({len(res):,} rows) and dev_ranges.csv ({len(ranges)} rows: "
      + ", ".join(f"{k} {n}" for k, n in ranges["kind"].value_counts().items()) + ")")

Saved dev_residuals.csv (9,343 rows) and dev_ranges.csv (75 rows: persistence 19, age 17, draft 12, age_x_draft 11, experience 10, luck_fit 3, group_mean 3)


## Takeaways
- **9,343 tested player-seasons** (5,523 forward, 3,039 defense, 781 goalie). The projection runs slightly high for skaters (average miss -0.005 WAR/60 for forwards, -0.004 for defensemen, 0.000 for goalies). Draws are centered so the simulated league doesn't drift down.
- **The biggest surprises are recognizable.** Up: Malkin 2011-12 (+5.0 WAR), Ovechkin 2009-10, Crosby 2018-19; Tim Thomas 2010-11 (+6.6) and Ullmark 2022-23 in goal. Down: Strome and Kuznetsov 2022-23; Grubauer 2021-22 (-5.9) and Price 2017-18 in goal.
- **Most of a surprise season is luck.** For a forward at 1,400 minutes about 55% is luck (68% at 800 minutes); defensemen 60% (73%); goalies 87% at 3,000 minutes. A typical miss is about 1.1 WAR for a forward at 1,400 minutes, 0.7 for a defenseman, and 2.2 for a goalie at 3,000.
- **Very little of a surprise sticks beyond what the blend already carries:** next season's miss correlates with this one at 0.03 (forwards), 0.06 (defensemen) and -0.03 (goalies, used as 0). So the non-luck part is mostly one-year change (role, linemates, health), not permanent change.
- **Age moves the range.** Forwards 21 and under beat their projection by +0.015 WAR/60 (centered) and forwards 33+ miss by -0.005; defensemen show the same pattern (+0.004 at 22-23, -0.005 at 33+). Young players' ranges are also wider and old forwards' narrower. Centered aging still under-ages, as notebook 15 warned.
- **Experience doesn't**, once age is in the buckets: no experience bucket differs from its group beyond noise.
- **Draft position does, for forwards and for the young.** Top-5 forwards beat their projection by +0.012 WAR/60, undrafted forwards miss by -0.012. Among skaters 23 and under, top-5 picks beat their projection by +0.022 at 21 and under (133 player-seasons, 72% beat it) and +0.009 at 22-23. No other age × draft cell stands out. So **top picks keep beating their projection even with NHL history**, fading by 22-23, the same pattern notebook 16 found for NHLe. Both top-5 cells get their own draw bucket.
- **NHLe adds something for young skaters.** For skaters 23 and under with 3 or fewer NHL seasons, best pre-NHL NHLe correlates 0.19 with the miss. The top fifth beat their projection by +0.021 WAR/60, the bottom fifth by -0.002. With age and draft already in, NHLe stays clearly positive (+0.058 per NHL point per game, 6.2 standard errors; R² 0.016 to 0.041). It belongs in the prospect rule.
- **Dropouts are the hidden downside.** About 10% of regular forwards, 12% of defensemen and 17% of goalies nearly vanish the next season (injury, demotion, Europe, retirement). It's 25-29% at 33+, vs. 6-9% for skaters under 27. They never get a miss, so **the ranges understate the downside, most for veterans**. No season stands out as odd.
- **Calibration: scaled ranges hold up better than raw ones.** On 2018-19 onward (ranges built only on earlier seasons), the centered, scaled ranges the simulator uses contain 87-88% of outcomes in the 5-95 range (target 90%), 76-78% in 10-90 (80%) and 45-48% in 25-75 (50%). Raw ranges miss more (average distance from target 0.049 vs. 0.023 scaled; centering costs 0.004). Raw ranges are far too narrow for low-minute skaters (66% in 10-90), which scaling fixes (77-79%). But scaling leaves high-minute skaters (74-75%) and 3,000+ minute goalies (72-73%) a bit narrow. Overall the ranges run slightly narrow in recent seasons.
- **The demo passes the hockey check.** Over 1,000 draws of 2026-27 WAR:
  - Schaefer (19, #1 pick) has more upside: 90th percentile 1.7 WAR above his projection, 10th percentile 1.1 below, median +0.5.
  - Hughes (27) is balanced (+1.1 / -1.1).
  - Karlsson (36) leans down (+0.9 / -1.1, median -0.15), and that's before counting dropouts.
- **Next:** minutes projections (who plays how much), then the season simulator. For the simulator: decide whether the game simulator uses only the non-luck parts, widen the ranges slightly or add a dropout draw for veterans, and build the prospect rule (NHLe plus draft position for players with little NHL history).